# PEFT runs on XNLI-bn — DoRA and IA3 (A100, fp16)

Run cells **top to bottom**. Nothing needs editing.

1. **Cell 1** installs pinned libraries (removes torchao). Run once, then **Runtime → Restart session**.
2. **Cells 2–4**: setup, data (XNLI-bn train/val/test + BNLI test as the second exam), engine. No training yet.
3. **Cell 5** = IA3 learning-rate check: 3 runs at 5% data, spare seed 0, rates 1e-3 / 3e-3 / 1e-2.
   Rule fixed **before** running: the rate with the highest **validation** macro-F1 wins. Test scores of these runs are never used.
   Note your compute-unit balance before and after. **Then stop and send the output to Claude.**
4. The main DoRA/IA3 grid cell is added after the next decision (which fractions/seeds).

If Colab disconnects: reconnect, run cells 2, 3, 4, then re-run the cell you were on. Finished runs are skipped.

In [ ]:
# ---- CELL 1: pinned libraries (run ONCE, then Runtime -> Restart session) ----
!pip uninstall -y -q torchao   # newest torchao breaks LoRA/DoRA in peft 0.14.0
!pip install -q "transformers==4.54.1" "peft==0.14.0"
!pip install -q "accelerate>=0.30,<1.5" "datasets>=2.19,<3.5" "evaluate>=0.4.1" "scikit-learn>=1.3" "scipy>=1.11"
!pip install -q git+https://github.com/csebuetnlp/normalizer
print("\nDone. Red 'dependency conflict' lines about diffusers/gradio/gcsfs are harmless.")
print("NOW: Runtime -> Restart session, then run cell 2.")

In [ ]:
# ---- CELL 2: Drive, settings, environment checks ----
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, importlib.util, torch, transformers, peft

LRCHECK_DIR  = "/content/drive/MyDrive/thesis_bangla_nli_ia3_lrcheck"   # IA3 rate check only
PEFT_DIR     = "/content/drive/MyDrive/thesis_bangla_nli_peft"          # main DoRA/IA3 results
BNLI_TEST    = "/content/drive/MyDrive/thesis_bangla_nli_bnli/bnli_test.csv"
SCRATCH_CKPT = "/content/ckpt"
PRECISION    = "fp16"            # all earlier runs used fp16
TRAIN_DATASET = "xnli_bn"
RESUME_INTERRUPTED = True
RETRIES      = 1
os.makedirs(SCRATCH_CKPT, exist_ok=True)

def load_engine(project_dir, ia3_lr=None):
    """Point the engine at a results folder and (re)load it."""
    global PROJECT_DIR, AUTO_EXPORT_DIR, IA3_LR
    PROJECT_DIR, AUTO_EXPORT_DIR, IA3_LR = project_dir, project_dir, ia3_lr
    os.makedirs(project_dir, exist_ok=True)
    exec(open("/content/peft_engine.py").read(), globals())

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("GPU:", gpu)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert "A100" in gpu, "Not an A100 - STOP. Change runtime type to A100."
assert transformers.__version__ == "4.54.1" and peft.__version__ == "0.14.0", "Wrong versions - run cell 1, then restart session."
assert importlib.util.find_spec("torchao") is None, "torchao is installed - run: !pip uninstall -y torchao, then restart session"
assert os.path.exists(BNLI_TEST), f"missing {BNLI_TEST}"
print("\nCELL 2 OK")

In [ ]:
# ---- CELL 3: data. Train on XNLI-bn (identical preprocessing to the original runs); second exam = BNLI test ----
from datasets import load_dataset
from transformers import AutoTokenizer
from normalizer import normalize
import numpy as np, evaluate, collections

xnli = load_dataset("csebuetnlp/xnli_bn", revision="refs/convert/parquet")
assert xnli["train"].features["label"].names == ["contradiction", "entailment", "neutral"], "XNLI-bn label order changed - STOP"
LABEL2ID = {"contradiction": 0, "entailment": 1, "neutral": 2}
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

def normalize_and_tokenize(examples):
    s1 = [normalize(t) for t in examples["sentence1"]]
    s2 = [normalize(t) for t in examples["sentence2"]]
    return tokenizer(s1, s2, truncation=True, padding="max_length", max_length=128)

tokenized_ds = {s: xnli[s].map(normalize_and_tokenize, batched=True) for s in ["train", "validation", "test"]}

bnli_test = load_dataset("csv", data_files={"test": BNLI_TEST})["test"]
bnli_test = bnli_test.map(lambda ex: {"sentence1": ex["premise"], "sentence2": ex["hypothesis"], "label_id": LABEL2ID[ex["label"]]})
bnli_test = bnli_test.remove_columns(["premise", "hypothesis", "label", "similarity"]).rename_column("label_id", "label")
other_test_ds = bnli_test.map(normalize_and_tokenize, batched=True)

accuracy_metric, f1_metric = evaluate.load("accuracy"), evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
            "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]}

sizes = {k: len(v) for k, v in tokenized_ds.items()}
print("XNLI-bn:", sizes, "| BNLI test (second exam):", len(other_test_ds))
assert sizes == {"train": 381449, "validation": 2419, "test": 4895}
got = sorted(collections.Counter(other_test_ds["label"]).items())
assert got == [(0, 756), (1, 748), (2, 748)], f"BNLI test label counts wrong: {got}"
print("\nCELL 3 OK")

In [ ]:
%%writefile /content/peft_engine.py
# PEFT training engine = Step10_retrain_ColabA100.ipynb cell 4 + BNLI edits + PEFT edits (DoRA, IA3).
# Needs from earlier cells: PROJECT_DIR, SCRATCH_CKPT, PRECISION, _free_gb, tokenized_ds, other_test_ds, compute_metrics, tokenizer.
# ---- cell 4: training engine (matched budgets, test scored in-run) ----
# =============================================================================
# STEP 10 v2 - retrain with MATCHED budgets and score the TEST split at train time
# =============================================================================
# What each design decision buys you (in plain English):
#   * both methods get the SAME epoch ceiling (8) and the same stopping rule
#     (none). Neither is truncated while still improving.
#   * the test split is scored INSIDE the run and its logits saved to a .npz,
#     so no post-hoc checkpoint can vanish and no accidental selection on test
#     can happen.
#   * per-example test logits enable McNemar / paired-bootstrap tests later
#     without paying for GPU again.
#   * trainable params, wall-clock, peak GPU memory, model file size are all
#     recorded - the efficiency claims LoRA is actually about.
#   * training checkpoints live on SCRATCH disk (/content/ckpt); only the final
#     best model, curves, predictions and manifest reach PROJECT_DIR on Drive.
#   * resume is EXPLICIT and gated by a settings-stamp: a rerun is a rerun, and
#     only a genuinely interrupted run continues where it stopped.
#
# [v2] additions vs your first pass:
#   * HeartbeatCallback writes runs_progress/<run_id>.json after every eval
#   * post-save reload verification (bit-exact drift written to manifest)
#   * automatic OOM recovery (retry with smaller batch + grad accum)
#   * disk-space guard before starting every run
# =============================================================================

import glob, json, math, os, shutil, time, gc
import numpy as np
import pandas as pd
import torch
import transformers, peft
from transformers import (
    AutoModelForSequenceClassification, EarlyStoppingCallback,
    Trainer, TrainingArguments, TrainerCallback,
    default_data_collator, set_seed
)
from peft import LoraConfig, IA3Config, TaskType, get_peft_model, PeftModel

# ---- configuration ---------------------------------------------------------
RESULTS_V2   = f"{PROJECT_DIR}/results_v2.csv"
RUNS_DIR     = f"{PROJECT_DIR}/runs"
CURVES_DIR   = f"{PROJECT_DIR}/curves_v2"
PREDS_DIR    = f"{PROJECT_DIR}/preds"
MODELS_DIR   = f"{PROJECT_DIR}/best_models"
PROGRESS_DIR = f"{PROJECT_DIR}/runs_progress"   # [v2] heartbeat manifests
SCRATCH_CKPT = globals().get("SCRATCH_CKPT", "/content/ckpt")

BASE_MODEL      = "csebuetnlp/banglabert"
NUM_LABELS      = 3
BATCH_SIZE      = 16          # keep identical to your first pass
EVAL_BATCH      = 64
CEILING         = 8           # SAME 8-epoch ceiling for BOTH methods
PATIENCE_EPOCHS = None        # None = NO early stopping (deliberate; see thesis §5.2)

# PRECISION was set in cell 1 based on the actual GPU. Read at call time so
# editing it above and re-running THIS cell alone takes effect.
PRECISION = globals().get("PRECISION", "bf16")

LR       = {"lora": 2e-4, "full_ft": 2e-5}
# PEFT edit: DoRA = LoRA's exact settings + use_dora=True (authors: LoRA config "can already achieve better results most of the time")
LR["dora"] = 2e-4
# PEFT edit: IA3 learning-rate check candidates (paper value 3e-3 +/- one step); final "ia3" rate is set by the notebook
IA3_LR_CANDIDATES = {"ia3lr0.001": 1e-3, "ia3lr0.003": 3e-3, "ia3lr0.01": 1e-2}
LR.update(IA3_LR_CANDIDATES)
if globals().get("IA3_LR") is not None:
    LR["ia3"] = IA3_LR
# PEFT edit: IA3 on keys, values (self-attention) and the FFN intermediate activation (input of the FFN output layer), as in Liu et al. 2022
IA3_TARGETS = r".*encoder\.layer\.\d+\.(attention\.self\.(key|value)|output\.dense)"
IA3_FF      = r".*encoder\.layer\.\d+\.output\.dense"
TRAIN_DATASET = globals().get("TRAIN_DATASET", "xnli_bn")
LORA_CFG = dict(r=8, lora_alpha=16, lora_dropout=0.1,
                target_modules=["query", "value"])

# Full-FT checkpoints are ~443 MB each; 25 runs would eat 11 GB on Drive.
# The test predictions + manifest already contain everything the thesis needs,
# so full-FT weights are measured (for the efficiency table) then deleted.
# LoRA adapters are ~3.6 MB, so keep those.
SAVE_BEST_MODEL = {"lora": True, "full_ft": False}

# Below this dev macro-F1 the run is treated as a collapse (fp16 underflow,
# NaN loss, bad LR) - no manifest is written, so run_grid retries it.
CHANCE_F1 = 0.40      # 3 classes -> random baseline ~0.33

# ---- unattended running: power cuts, crashes, deadlines --------------------
RESUME_INTERRUPTED = globals().get("RESUME_INTERRUPTED", True)
RETRIES            = globals().get("RETRIES", 1)
AUTO_EXPORT_DIR    = globals().get("AUTO_EXPORT_DIR", None)

# Fallback speed numbers for the time estimator, measured on Tesla T4/bs=16/fp16.
# The estimator replaces these with measurements from THIS GPU once one full run
# of each method has finished here.
SPEED_FALLBACK = {"full_ft": 87.5, "lora": 154.3}

# [v2] Auto-export cadence: dump a fresh zip every N finished runs, not every run.
# Every run is safe but slow; every 3 keeps Drive fresh with much less overhead.
AUTO_EXPORT_EVERY = 3

# ---- one-shot precision/GPU sanity check -----------------------------------
def check_precision():
    if PRECISION not in ("fp32", "fp16", "bf16"):
        raise ValueError(f"PRECISION must be fp32|fp16|bf16 - got {PRECISION!r}")
    if PRECISION == "bf16" and torch.cuda.is_available() \
            and not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{torch.cuda.get_device_name(0)} cannot do bf16 - "
                           "set PRECISION='fp16' in cell 1 and re-run cell 4.")

def evals_per_epoch(fraction):
    """1 eval/epoch at low fractions (short epochs), 2 at 10%+ (longer epochs)."""
    return 1   # BNLI edit: was `1 if fraction <= 0.10 else 2`; BNLI uses fraction=1.0, keep XNLI protocol

# ---- dataset helpers -------------------------------------------------------
KEEP = ["input_ids", "attention_mask", "token_type_ids", "label"]

def _strip(ds):
    return ds.remove_columns([c for c in ds.column_names if c not in KEEP])

def _dir_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        total += sum(os.path.getsize(os.path.join(root, f)) for f in files)
    return round(total / 1e6, 2)

def _mkdirs():
    for d in (RUNS_DIR, CURVES_DIR, PREDS_DIR, MODELS_DIR,
              PROGRESS_DIR, SCRATCH_CKPT):
        os.makedirs(d, exist_ok=True)

def _free_gb(path):
    return round(shutil.disk_usage(path).free / 1e9, 2)

def this_gpu():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

def build_model(method, seed):
    """(model, trainable, total). set_seed BEFORE construction so classifier
    head init and LoRA A/B matrices are reproducible seed-by-seed."""
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS)
    if method == "lora":
        model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_CLS, **LORA_CFG))
    elif method == "dora":                                   # PEFT edit
        model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_CLS, use_dora=True, **LORA_CFG))
    elif method.startswith("ia3"):                           # PEFT edit
        model = get_peft_model(base, IA3Config(task_type=TaskType.SEQ_CLS,
                                               target_modules=IA3_TARGETS, feedforward_modules=IA3_FF))
    elif method == "full_ft":
        model = base
    else:
        raise ValueError(f"unknown method {method!r}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return model, trainable, total

def rebuild_results_csv():
    """results_v2.csv is DERIVED from runs/*.json - never edit it by hand."""
    rows = [json.load(open(p)) for p in sorted(glob.glob(f"{RUNS_DIR}/*.json"))]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(["fraction", "seed", "method"])
    df.to_csv(RESULTS_V2, index=False)
    return df

def done_runs():
    return {os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(f"{RUNS_DIR}/*.json")}

# ---- moving results between machines (unchanged from v1) -------------------
TRANSFER_DIRS = ("runs", "curves_v2", "preds", "best_models")

def import_zip(zip_path, apply=True):
    """Merge another machine's zip into this PROJECT_DIR. Nothing is overwritten
    - a duplicate byte-identical file is skipped, a differing one is saved as
    .incoming beside the local copy so you can pick manually."""
    import zipfile
    _mkdirs()
    added = same = clash = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            if info.is_dir(): continue
            parts = info.filename.replace("\\", "/").split("/")
            at = [i for i, p in enumerate(parts[:-1]) if p in TRANSFER_DIRS]
            if not at: continue
            rel = "/".join(parts[at[-1]:])
            dest = f"{PROJECT_DIR}/{rel}"
            data = z.read(info)
            if os.path.exists(dest):
                if open(dest, "rb").read() == data:
                    same += 1
                else:
                    clash += 1
                    print(f"  [!] {rel} differs - local kept, incoming saved as .incoming")
                    if apply:
                        open(dest + ".incoming", "wb").write(data)
                continue
            added += 1
            if apply:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                open(dest, "wb").write(data)
    print(f"import: {added} new, {same} identical, {clash} conflict")
    if added and apply:
        rebuild_results_csv()
    return added, same, clash

def export_zip(out_dir=None, include_models=False, name=None, quiet=False):
    """Zip runs/, curves_v2/, preds/ for backup. Safe to re-import."""
    import zipfile
    out_dir = out_dir or PROJECT_DIR
    dirs = TRANSFER_DIRS if include_models else tuple(
        d for d in TRANSFER_DIRS if d != "best_models")
    tag = (torch.cuda.get_device_name(0).split()[-1]
           if torch.cuda.is_available() else "cpu")
    out = os.path.join(out_dir, name or
                       f"thesis_runs_{tag}_{time.strftime('%Y%m%d_%H%M')}.zip")
    tmp, n = out + ".part", 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for name_ in dirs:
            for p in sorted(glob.glob(f"{PROJECT_DIR}/{name_}/**/*", recursive=True)):
                if os.path.isfile(p):
                    z.write(p, os.path.relpath(p, PROJECT_DIR).replace("\\", "/"))
                    n += 1
    os.replace(tmp, out)   # atomic move so a crash mid-zip cannot leave junk
    if not quiet:
        print(f"wrote {out}  ({n} files, {round(os.path.getsize(out)/1e6, 2)} MB)")
    return out

# ---- settings stamp (guarantees fair comparison) ---------------------------
def settings_now(method, fraction, ceiling=None):
    return {"precision": PRECISION,
            "epoch_ceiling": CEILING if ceiling is None else ceiling,
            "batch_size": BATCH_SIZE, "patience_epochs": PATIENCE_EPOCHS,
            "learning_rate": LR[method], "evals_per_epoch": evals_per_epoch(fraction)}

def stale_runs(fractions, seeds, methods=("full_ft", "lora"), ceiling=None):
    out = []
    for f in sorted(fractions):
        for s in seeds:
            for m in methods:
                run_id = f"{m}_frac{f}_seed{s}"
                p = f"{RUNS_DIR}/{run_id}.json"
                if not os.path.exists(p): continue
                saved, now = json.load(open(p)), settings_now(m, f, ceiling)
                diff = {k: (saved.get(k), v) for k, v in now.items()
                        if saved.get(k) != v}
                if diff: out.append((run_id, diff))
    return out

def report_stale(fractions, seeds, methods=("full_ft", "lora"), ceiling=None):
    stale = stale_runs(fractions, seeds, methods, ceiling)
    for run_id, diff in stale:
        detail = ", ".join(f"{k}: {w!r} -> {n!r}" for k, (w, n) in diff.items())
        print(f"  [STALE] {run_id} was trained with {detail}")
    if stale:
        print(f"  {len(stale)} finished run(s) no longer match. Re-run with force=True.")

def report_split_risk(fractions, seeds, methods=("full_ft", "lora")):
    """Warn BEFORE training when a pair (same fraction+seed, both methods) would
    end up split across two GPUs. The per-seed gap must come from one GPU."""
    here, risky = this_gpu(), []
    for f in sorted(fractions):
        for s in seeds:
            saved = {}
            for m in methods:
                p = f"{RUNS_DIR}/{m}_frac{f}_seed{s}.json"
                if os.path.exists(p):
                    saved[m] = json.load(open(p)).get("gpu", "?")
            todo = [m for m in methods if m not in saved]
            other = {g for g in saved.values() if g != here}
            if todo and other:
                risky.append((f, s, todo, sorted(other)))
    for f, s, todo, other in risky:
        print(f"  [SPLIT RISK] fraction {f:g} seed {s}: {todo} would run here "
              f"on {here}, partner already on {other}")
    return risky

# ---- resume-from-checkpoint (unchanged from v1 core; robust) ---------------
class ResumeFailed(RuntimeError):
    """Saved checkpoint could not be read back - usually power cut mid-write."""

def _stamp(method, fraction, seed, ceiling, n_train):
    s = dict(settings_now(method, fraction, ceiling))
    s.update(run_id=f"{method}_frac{fraction}_seed{seed}", seed=seed, n_train=n_train,
             base_model=BASE_MODEL,
             lora_cfg=LORA_CFG if method in ("lora", "dora") else None,
             use_dora=(method == "dora"),
             ia3_cfg=dict(target_modules=IA3_TARGETS, feedforward_modules=IA3_FF) if method.startswith("ia3") else None)
    return s

def _resume_from(run_id, stamp):
    d = f"{SCRATCH_CKPT}/{run_id}"
    cks = sorted((p for p in glob.glob(f"{d}/checkpoint-*") if os.path.isdir(p)),
                 key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True)
    if not cks: return None, 0
    sp = f"{d}/_settings_stamp.json"
    saved = json.load(open(sp)) if os.path.exists(sp) else None
    if saved != stamp:
        why = "has no settings stamp" if saved is None else "was trained under " + \
              ", ".join(f"{k}: {saved.get(k)!r} -> {v!r}"
                        for k, v in stamp.items() if saved.get(k) != v)
        print(f"  [FRESH START] leftover checkpoint for {run_id} {why} - deleting.")
        shutil.rmtree(d, ignore_errors=True)
        return None, 0
    for p in cks:
        if os.path.exists(f"{p}/trainer_state.json"):
            step = int(p.rsplit("-", 1)[1])
            print(f"  [RESUME] {run_id} continuing from step {step}. Wall-clock "
                  f"for this run is not comparable and is excluded from timing.")
            return p, step
    print(f"  [FRESH START] {run_id} has partial checkpoints - deleting them.")
    shutil.rmtree(d, ignore_errors=True)
    return None, 0

# ---- [v2] HeartbeatCallback: mid-run progress visible from Drive ------------
class HeartbeatCallback(TrainerCallback):
    """Writes a tiny JSON after every evaluation into runs_progress/<run_id>.json.
    Contains current step/epoch, best-so-far F1 and clock time. If Colab kills
    the tab you know exactly how far the run got just by opening Drive."""
    def __init__(self, run_id, method, fraction, seed):
        self.run_id, self.method = run_id, method
        self.fraction, self.seed = fraction, seed
        self.t0 = time.time()
        self.best = -1.0
        self.path = f"{PROGRESS_DIR}/{run_id}.json"

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None: return
        f1 = metrics.get("eval_f1_macro", -1.0)
        if f1 > self.best: self.best = f1
        json.dump({
            "run_id": self.run_id, "method": self.method,
            "fraction": self.fraction, "seed": self.seed,
            "step": int(state.global_step), "epoch": float(state.epoch or 0),
            "elapsed_min": round((time.time() - self.t0) / 60, 2),
            "last_eval_f1": float(f1), "best_f1_so_far": float(self.best),
            "gpu": this_gpu(), "updated_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, open(self.path, "w"), indent=2)

    def on_train_end(self, args, state, control, **kwargs):
        if os.path.exists(self.path):
            os.remove(self.path)   # tidy: heartbeat gone once the run is done

# ---- speed & planning (unchanged core) -------------------------------------
def _full_train_size():
    try: return len(tokenized_ds["train"])
    except: return 381449

def measured_speed(method):
    here, vals = this_gpu(), []
    for p in glob.glob(f"{RUNS_DIR}/*.json"):
        r = json.load(open(p))
        if r.get("method") == method and r.get("gpu") == here \
                and r.get("wall_clock_comparable", True) \
                and r.get("train_samples_per_s"):
            vals.append(r["train_samples_per_s"])
    return (float(np.mean(vals)), len(vals)) if vals \
        else (SPEED_FALLBACK.get(method, SPEED_FALLBACK['lora']), 0)

def estimate_minutes(method, fraction, ceiling=None):
    ceiling = CEILING if ceiling is None else ceiling
    sps, _ = measured_speed(method)
    n_train = int(_full_train_size() * fraction)
    n_eval = 2419 * ceiling * evals_per_epoch(fraction) + 4895
    return (ceiling * n_train / sps + n_eval / (3 * sps)) / 60

def plan(fractions, seeds, methods=("full_ft", "lora"), ceiling=None,
         deadline=None, hours_per_day=24.0):
    rows = []
    for f in sorted(fractions):
        for s in seeds:
            for m in methods:
                run_id = f"{m}_frac{f}_seed{s}"
                done = os.path.exists(f"{RUNS_DIR}/{run_id}.json")
                mins = 0.0 if done else estimate_minutes(m, f, ceiling)
                rows.append(dict(run_id=run_id, fraction=f, seed=s, method=m,
                                 status="done" if done else "todo",
                                 est_min=round(mins, 1)))
    df = pd.DataFrame(rows)
    df["cum_h"] = (df["est_min"].cumsum() / 60).round(2)
    todo = df[df.status == "todo"]
    for m in methods:
        sps, n = measured_speed(m)
        print(f"  {m:8s}: {sps:6.1f} samples/s "
              + (f"({n} run(s) here)" if n else "(T4 fallback)"))
    print(f"\n  {len(todo)} to do, {todo.est_min.sum()/60:.1f} GPU-h; "
          f"{len(df)-len(todo)} done")
    return df

def auto_export():
    if not AUTO_EXPORT_DIR: return None
    try:
        os.makedirs(AUTO_EXPORT_DIR, exist_ok=True)
        tag = this_gpu().split()[-1]
        return export_zip(AUTO_EXPORT_DIR,
                          name=f"thesis_runs_{tag}_latest.zip", quiet=True)
    except Exception as e:
        print(f"  [!] auto-export failed ({type(e).__name__}: {e})")
        return None

# ---- [v2] post-save reload verification ------------------------------------
def _reload_and_verify(save_dir, method, test_ds, expected_logits):
    """Reload the saved model from disk and re-score the test set.
    Compare against the in-memory predictions. Returns max abs logit drift.
    A drift of 0.000000 (bit-exact) means load_best_model_at_end + trainer.save_model
    round-tripped cleanly; anything above 1e-3 flags a serialization bug."""
    try:
        if method != "full_ft":                              # PEFT edit: any adapter method
            base = AutoModelForSequenceClassification.from_pretrained(
                BASE_MODEL, num_labels=NUM_LABELS)
            reloaded = PeftModel.from_pretrained(base, save_dir)
        else:
            reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
        reloaded.eval()
        if torch.cuda.is_available(): reloaded = reloaded.cuda()
        # Run through the Trainer's predict path for consistency
        v_trainer = Trainer(model=reloaded, args=TrainingArguments(
            output_dir="/tmp/_verify", per_device_eval_batch_size=EVAL_BATCH,
            report_to="none",
            fp16=(PRECISION == "fp16"), bf16=(PRECISION == "bf16")),
            data_collator=default_data_collator)
        v_pred = v_trainer.predict(test_ds)
        drift = float(np.max(np.abs(v_pred.predictions.astype(np.float32)
                                    - expected_logits.astype(np.float32))))
        del reloaded, v_trainer
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return drift, True
    except Exception as e:
        print(f"  [!] reload verification failed: {type(e).__name__}: {e}")
        return None, False

# ---- the actual run -------------------------------------------------------
def run_one(method, fraction, seed, ceiling=None, force=False,
            _oom_batch=None, _oom_accum=1):
    """One run start to finish. train -> select on dev -> score test ->
       reload+verify -> log. About 5-15 min on A100 at fractions 0.01-0.05.
       _oom_batch / _oom_accum are used only by the OOM auto-retry path."""
    ceiling = CEILING if ceiling is None else ceiling
    run_id = f"{method}_frac{fraction}_seed{seed}"
    manifest_path = f"{RUNS_DIR}/{run_id}.json"

    # Skip if already finished, unless settings changed since then.
    if os.path.exists(manifest_path) and not force:
        saved = json.load(open(manifest_path))
        diff = {k: (saved.get(k), v)
                for k, v in settings_now(method, fraction, ceiling).items()
                if saved.get(k) != v}
        if diff:
            print(f"[SKIP] {run_id} already done under DIFFERENT settings: "
                  + ", ".join(f"{k} {w!r}->{n!r}" for k, (w, n) in diff.items())
                  + " - re-run with force=True.")
        else:
            print(f"[SKIP] {run_id} already done.")
        return saved

    _mkdirs()
    check_precision()

    # [v2] disk-space guard: don't start a run that will fail on save
    if _free_gb(SCRATCH_CKPT) < 3:
        raise RuntimeError(f"scratch disk has {_free_gb(SCRATCH_CKPT)} GB free "
                           f"- need at least 3. Free /content and re-run.")
    set_seed(seed)

    n_train = int(len(tokenized_ds["train"]) * fraction)
    # Identical subset for both methods at a given seed; nested as fraction grows.
    train_ds = _strip(tokenized_ds["train"].shuffle(seed=seed).select(range(n_train)))
    dev_ds   = _strip(tokenized_ds["validation"])
    test_ds  = _strip(tokenized_ds["test"])

    effective_batch = _oom_batch or BATCH_SIZE
    steps_per_epoch = max(1, math.ceil(n_train / effective_batch))
    per_epoch = evals_per_epoch(fraction)
    eval_every = max(1, steps_per_epoch // per_epoch)
    patience = PATIENCE_EPOCHS * per_epoch if PATIENCE_EPOCHS else None
    by_steps = per_epoch > 1

    model, trainable, total = build_model(method, seed)
    print(f"[RUN] {run_id}  n_train={n_train}  ceiling={ceiling} epochs  "
          f"lr={LR[method]}  eval every {eval_every} steps  "
          f"{'patience=%d evals' % patience if patience else 'no early stopping'}  "
          f"trainable={trainable:,}/{total:,} ({100*trainable/total:.2f}%)")
    if _oom_batch:
        print(f"  [OOM RETRY] using batch={_oom_batch}, grad_accum={_oom_accum} "
              f"(effective batch preserved at {BATCH_SIZE})")

    # Continue from a genuine interruption if one exists and matches settings.
    stamp = _stamp(method, fraction, seed, ceiling, n_train)
    ckpt_dir = f"{SCRATCH_CKPT}/{run_id}"
    resume_path, resume_step = _resume_from(run_id, stamp) \
        if RESUME_INTERRUPTED else (None, 0)
    os.makedirs(ckpt_dir, exist_ok=True)
    json.dump(stamp, open(f"{ckpt_dir}/_settings_stamp.json", "w"), indent=2)

    args = TrainingArguments(
        output_dir=f"{SCRATCH_CKPT}/{run_id}",
        num_train_epochs=ceiling,
        learning_rate=LR[method],
        per_device_train_batch_size=effective_batch,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=_oom_accum,
        eval_strategy="steps" if by_steps else "epoch",
        save_strategy="steps" if by_steps else "epoch",
        **({"eval_steps": eval_every, "save_steps": eval_every} if by_steps else {}),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed, data_seed=seed,
        fp16=(PRECISION == "fp16"),
        bf16=(PRECISION == "bf16"),
        logging_steps=max(10, eval_every // 2),
        report_to="none",
    )

    callbacks = [HeartbeatCallback(run_id, method, fraction, seed)]
    if patience:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=patience))

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=dev_ds,
                      compute_metrics=compute_metrics,
                      data_collator=default_data_collator,
                      callbacks=callbacks)

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        train_out = trainer.train(resume_from_checkpoint=resume_path) \
            if resume_path else trainer.train()
    except torch.cuda.OutOfMemoryError as e:
        # [v2] OOM auto-retry: halve the batch, double grad_accum, restart clean
        if _oom_batch and _oom_batch <= 4:
            raise
        new_batch = (_oom_batch or BATCH_SIZE) // 2
        new_accum = _oom_accum * 2
        print(f"  [OOM] retrying with batch={new_batch}, grad_accum={new_accum}")
        del model, trainer
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        return run_one(method, fraction, seed, ceiling, force=True,
                       _oom_batch=new_batch, _oom_accum=new_accum)
    except Exception as e:
        if not resume_path:
            raise
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        raise ResumeFailed(
            f"[{run_id}] could not continue from {os.path.basename(resume_path)}: "
            f"{type(e).__name__}: {e}. Checkpoint deleted; next attempt is fresh."
        ) from e

    wall = time.time() - t0
    peak_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1) \
        if torch.cuda.is_available() else None

    # ---- what happened, epoch by epoch ----------------------------------
    curve = [l for l in trainer.state.log_history if "eval_f1_macro" in l]
    if not curve:
        raise RuntimeError(f"[{run_id}] no eval_f1_macro in log_history")
    best = max(curve, key=lambda x: x["eval_f1_macro"])
    json.dump(curve, open(f"{CURVES_DIR}/{run_id}.json", "w"), indent=2)
    if best["eval_f1_macro"] < CHANCE_F1:
        nan_seen = any(isinstance(l.get("loss"), float) and math.isnan(l["loss"])
                       for l in trainer.state.log_history)
        raise RuntimeError(
            f"[{run_id}] best dev macro-F1 {best['eval_f1_macro']:.4f} at chance"
            f"{' + NaN loss' if nan_seen else ''} - collapsed run, no manifest.")
    evals_ran = len(curve)
    evals_possible = ceiling * per_epoch
    early_stopped = evals_ran < evals_possible
    ceiling_hit = abs(float(best["epoch"]) - ceiling) < 1e-6

    # ---- dev at the loaded best model: must match the best curve point --
    dev = trainer.evaluate(dev_ds, metric_key_prefix="dev")
    drift = abs(dev["dev_f1_macro"] - best["eval_f1_macro"])
    if drift > 1e-3:
        print(f"  [!] dev F1 drift on reload: {drift:.4f} - "
              f"load_best_model_at_end not restoring what you think")

    # ---- test: scored ONCE, never used for selection ---------------------
    pred = trainer.predict(test_ds, metric_key_prefix="test")
    np.savez_compressed(f"{PREDS_DIR}/{run_id}_test.npz",
                        logits=pred.predictions.astype(np.float16),
                        labels=np.asarray(pred.label_ids))

    # BNLI edit: second exam - same model, the other dataset's test set (XNLI-bn)
    xpred = trainer.predict(_strip(other_test_ds), metric_key_prefix="xtest")
    np.savez_compressed(f"{PREDS_DIR}/{run_id}_xtest.npz",
                        logits=xpred.predictions.astype(np.float16),
                        labels=np.asarray(xpred.label_ids))

    # ---- the best model: weighed always, kept only if asked --------------
    keep_model = SAVE_BEST_MODEL.get(method, True)
    save_dir = f"{MODELS_DIR}/{run_id}" if keep_model \
        else f"{SCRATCH_CKPT}/_size_probe/{run_id}"
    shutil.rmtree(save_dir, ignore_errors=True)
    trainer.save_model(save_dir)
    saved_mb = _dir_mb(save_dir)

    # [v2] reload verification: load from disk, re-score test, compare logits
    reload_drift, reload_ok = _reload_and_verify(
        save_dir, method, test_ds, pred.predictions)

    if not keep_model:
        shutil.rmtree(save_dir, ignore_errors=True)
        shutil.rmtree(f"{MODELS_DIR}/{run_id}", ignore_errors=True)

    manifest = {
        "run_id": run_id, "method": method, "fraction": fraction, "seed": seed,
        "n_train": n_train, "epoch_ceiling": ceiling,
        "best_epoch": float(best["epoch"]),
        "best_step": int(best.get("step", 0)),
        "evals_ran": evals_ran, "evals_possible": evals_possible,
        "early_stopped": bool(early_stopped), "ceiling_hit": bool(ceiling_hit),
        "patience_evals": patience, "patience_epochs": PATIENCE_EPOCHS,
        "evals_per_epoch": per_epoch,
        "learning_rate": LR[method], "batch_size": BATCH_SIZE,
        "effective_batch": effective_batch * _oom_accum,
        "grad_accum_steps": _oom_accum,
        "precision": PRECISION,
        "dev_accuracy": float(dev["dev_accuracy"]),
        "dev_f1_macro": float(dev["dev_f1_macro"]),
        "dev_f1_from_curve": float(best["eval_f1_macro"]),
        "dev_reload_drift": float(drift),
        "test_accuracy": float(pred.metrics["test_accuracy"]),
        "test_f1_macro": float(pred.metrics["test_f1_macro"]),
        "xtest_accuracy": float(xpred.metrics["xtest_accuracy"]),     # BNLI edit
        "xtest_f1_macro": float(xpred.metrics["xtest_f1_macro"]),     # BNLI edit
        "train_dataset": TRAIN_DATASET,                               # PEFT edit
        "trainable_params": int(trainable), "total_params": int(total),
        "pct_trainable": round(100 * trainable / total, 4),
        "train_runtime_s": round(wall, 1),
        "train_samples_per_s": round(
            train_out.metrics.get("train_samples_per_second", 0), 2),
        "resumed_from_step": int(resume_step),
        "wall_clock_comparable": not resume_step,
        "peak_gpu_mem_mb": peak_mb, "saved_model_mb": saved_mb,
        "best_model_kept": bool(keep_model),
        "reload_verified": bool(reload_ok),
        "reload_max_logit_drift": (round(reload_drift, 8)
                                   if reload_drift is not None else None),
        "gpu": this_gpu(),
        "transformers": transformers.__version__, "peft": peft.__version__,
        "torch": torch.__version__,
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    rebuild_results_csv()

    print(f"  dev F1 {manifest['dev_f1_macro']:.4f} @ epoch {manifest['best_epoch']:g}"
          f"{'  (early stopped)' if early_stopped else ''}"
          f"{'  (CEILING HIT)' if ceiling_hit else ''}"
          f"  ->  TEST acc {manifest['test_accuracy']:.4f} / "
          f"F1 {manifest['test_f1_macro']:.4f}   "
          f"[{wall/60:.1f} min, {peak_mb} MB peak, model {saved_mb} MB, "
          f"reload drift {manifest['reload_max_logit_drift']}]")

    del model, trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    shutil.rmtree(f"{SCRATCH_CKPT}/{run_id}", ignore_errors=True)
    return manifest


def run_grid(fractions, seeds, methods=("full_ft", "lora"), ceiling=None,
             force=False, retries=None, stop_after_hours=None, deadline=None):
    """Cheapest fractions first. Every finished run is saved to Drive before the
    next starts - re-running this cell after a disconnect just continues.
    Safe to leave unattended: transient failures are retried, auto-export writes
    a fresh backup zip periodically, and Ctrl-C stops between runs cleanly."""
    retries = RETRIES if retries is None else retries
    todo = [(f, s, m) for f in sorted(fractions) for s in seeds for m in methods]
    print(f"{len(todo)} run(s) queued; {len(done_runs())} already finished")
    report_stale(fractions, seeds, methods, ceiling)
    report_split_risk(fractions, seeds, methods)
    budget_h = stop_after_hours
    if deadline:
        left = (time.mktime(time.strptime(deadline, "%Y-%m-%d %H:%M"))
                - time.time()) / 3600
        budget_h = left if budget_h is None else min(budget_h, left)
        print(f"deadline {deadline}: {left:.1f} hours from now")
    if AUTO_EXPORT_DIR:
        print(f"auto-export every {AUTO_EXPORT_EVERY} run(s) -> {AUTO_EXPORT_DIR}")
    print()

    t_start, ok, failed, skipped = time.time(), 0, [], []
    for i, (f, s, m) in enumerate(todo):
        run_id = f"{m}_frac{f}_seed{s}"
        fresh = not os.path.exists(f"{RUNS_DIR}/{run_id}.json") or force
        if budget_h is not None and fresh:
            spent = (time.time() - t_start) / 3600
            est = estimate_minutes(m, f, ceiling) / 60
            if spent + est > budget_h:
                skipped.append(run_id); continue
        for attempt in range(retries + 1):
            try:
                run_one(m, f, s, ceiling=ceiling, force=force)
                if fresh:
                    ok += 1
                    if ok % AUTO_EXPORT_EVERY == 0:
                        auto_export()
                break
            except KeyboardInterrupt:
                print(f"\n[STOPPED] interrupted during {run_id}. Re-run this "
                      f"cell and it continues cleanly.")
                auto_export()
                return rebuild_results_csv()
            except Exception as e:
                last = attempt == retries
                print(f"[FAILED] {run_id}: {type(e).__name__}: {e}"
                      + ("" if last else f"  - retry {attempt+2}/{retries+1}"))
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                if last:
                    failed.append(run_id)
                    with open(f"{PROJECT_DIR}/failures.log", "a") as fh:
                        fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}\t"
                                 f"{run_id}\t{type(e).__name__}: {e}\n")

    # Final export at end of grid
    auto_export()
    spent = (time.time() - t_start) / 3600
    print(f"\n{ok} run(s) finished this session in {spent:.1f} h; "
          f"{len(done_runs())} total in folder")
    if failed:
        print(f"  {len(failed)} failed after retries: {', '.join(failed)}")
    if skipped:
        print(f"  {len(skipped)} skipped (out of time): {', '.join(skipped)}")
    return rebuild_results_csv()

print("training engine ready. run cell 5 (smoke test) first, then cell 6 (pilot).")


## STOP before cell 5
Write down your **compute-unit balance**. Run cell 5 (about 30–40 min). Write the balance again.
Send Claude: the full output of cell 5 + both balances.

In [ ]:
# ---- CELL 5: IA3 learning-rate check (validation only). Rule fixed in advance: highest validation macro-F1 wins ----
import json, pandas as pd
load_engine(LRCHECK_DIR)
cands = tuple(IA3_LR_CANDIDATES)                      # ("ia3lr0.001", "ia3lr0.003", "ia3lr0.01")
df = run_grid([0.05], [0], methods=cands)              # seed 0 = spare seed, not one of the 5 evaluation seeds
df = df[df.method.isin(cands)].copy()
df["lr"] = df.method.map(IA3_LR_CANDIDATES)
print("\n=== IA3 learning-rate check (VALIDATION scores only) ===")
print(df[["lr", "best_epoch", "ceiling_hit", "dev_f1_macro", "train_runtime_s", "trainable_params"]]
      .sort_values("lr").to_string(index=False))
missing = set(cands) - set(df.method)
if missing:
    print("\n[!] these candidates produced no result (collapsed or failed):", missing)
best = df.sort_values("dev_f1_macro", ascending=False).iloc[0]
choice = {"ia3_lr": float(best["lr"]), "dev_f1_macro": float(best["dev_f1_macro"]),
          "rule": "highest validation macro-F1 at 5% data, seed 0"}
json.dump(choice, open(f"{LRCHECK_DIR}/ia3_lr_choice.json", "w"), indent=2)
print("\nCHOSEN IA3 learning rate:", choice)